# Read and Process Data

In [0]:
# Import necessary PySpark libraries
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Initialize Spark session
spark = SparkSession.builder.appName("Spark DataFrames").getOrCreate()

In [0]:
# Read customers data from CSV file
# inferSchema=True automatically detects column data types
data = spark.read.csv('/Volumes/workspace/default/dataset/raw/customers.csv', header=True, inferSchema=True)
data.show()


In [0]:
# Count total number of customer records
data.count()

In [0]:
# Display the schema (column names and data types) of the DataFrame
data.printSchema()

In [0]:
# Create a working copy of the data
customers_df = data

# Convert registration_date string column to proper date type
customers_df = data.withColumn(
    "registration_date",
    to_date(col("registration_date"), "yyyy-MM-dd")
)

In [0]:
# Data Cleaning: Fill missing values in location columns with 'Unknown'
# This handles NULL values in city, state, and country fields
customers_df = customers_df.fillna(
    {
        'city': 'Unknown',
        'state': 'Unknown',
        'country': 'Unknown'
    }
)


In [0]:

# Feature Engineering: Extract year and month from registration_date
# This enables time-based analysis and aggregations
customers_df = customers_df.withColumn(
    "registration_year",
    year(col("registration_date"))
)\
.withColumn(
    "registration_month",
    month(col("registration_date"))
)

customers_df.show()

In [0]:
# Exploratory Data Analysis: Count unique values in location columns
# This shows the geographic diversity of the customer base
customers_df.select(
    countDistinct("city").alias("unique_cities")
).show()
customers_df.select(
    countDistinct("state").alias("unique_states")
).show()
customers_df.select(
    countDistinct("country").alias("unique_countries")
).show()


In [0]:
# Find top 5 cities with the most customers
customers_df.groupBy('city').count().orderBy(desc('count')).show(5)

In [0]:
# Alternative approach: Top 5 cities by customer count (using col() method)
customers_df.groupBy("city").count().orderBy(col('count').desc()).show(5)

In [0]:
# Find top 5 state-country combinations with most customers
# Useful for understanding geographic distribution patterns
customers_df.groupBy("state", "country").count().orderBy(col('count').desc()).show(5)

In [0]:
# Create a pivot table showing active vs inactive users by state
# Columns will be the unique values from is_active (True/False)
customers_df.groupBy("state").pivot("is_active").count().show()

In [0]:
# Window Functions: Rank customers by registration date within each state
# rank: leaves gaps after ties, dense_rank: no gaps, row_number: unique sequential numbers
window_spec = Window.partitionBy('state').orderBy(col('registration_date').desc())
customers_df = customers_df.withColumn('rank', rank().over(window_spec))\
                .withColumn('dense_rank', dense_rank().over(window_spec)) \
                .withColumn('row_number', row_number().over(window_spec))
                

In [0]:
# Display the ranking results to compare different ranking functions
customers_df.select("name", "state", "is_active", "rank", "dense_rank", "row_number").show()

In [0]:
# Filter customers who registered on or after January 1, 2025
# This creates a subset for analyzing recent customer acquisition
recent_customers = customers_df.filter(col("registration_date") >= lit("2025-01-01"))
recent_customers.show()

In [0]:
# Count how many customers registered since 2025
recent_customers.count()

In [0]:
# Find the earliest and latest registration dates for each city
# This shows the customer acquisition timeline per location
customers_df.groupBy("city").agg(
    min("registration_date").alias("oldest_customer"), 
    max("registration_date").alias("newest_customer")
).show()

In [0]:
# Save processed DataFrames to Parquet format for efficient storage and future use
# mode="overwrite" replaces existing data if present
output_path = '/Volumes/workspace/default/dataset'
customers_df.write.mode("overwrite").parquet(output_path + "/processed_customers")
recent_customers.write.mode("overwrite").parquet(output_path + "/recent_customers")


# Join Customers Orders with Customers_df 

In [0]:
# Display first 5 rows of the customers DataFrame
customers_df.display(5)

In [0]:
# Read orders data from CSV file
orders_df = spark.read.csv('/Volumes/workspace/default/dataset/raw/orders.csv', header=True, inferSchema=True)
orders_df.show(5)

In [0]:
# Extract month from order_date for time-based analysis
orders_df = orders_df.withColumn('order_month', month(col('order_date')))
orders_df.show(5)

In [0]:
# Join customers and orders DataFrames on customer_id (inner join)
# This combines customer information with their order history
customers_orders_df =  customers_df.join(orders_df, 'customer_id', 'inner')
customers_orders_df.show(5)

In [0]:
# Display customers DataFrame again for verification
customers_df.display(5)

In [0]:
# Calculate total number of orders per customer
# Orders are grouped by customer_id and sorted in descending order
customers_orders_count = customers_orders_df.groupBy('customer_id').count().orderBy(col('count').desc())
customers_orders_count.show(10)

In [0]:
# Calculate total spend per customer by summing all order amounts
# Results are sorted to show highest-spending customers first
customer_total_spend = customers_orders_df.groupBy('customer_id').agg(
    sum('total_amount').alias('total_spend')
).orderBy(col('total_spend').desc())
customer_total_spend.show(10)

# Store reference to top customers (for potential further analysis)
top_customers = customer_total_spend

In [0]:
# Calculate average order value per customer
# This shows the typical spending amount per order for each customer
customer_average_spend = customers_orders_df.groupBy('customer_id').agg(
    avg('total_amount').alias('total_spend')
).orderBy(col('total_spend').desc())
customer_average_spend.show(10)

In [0]:
# Count orders by their status (e.g., completed, pending, cancelled)
# This helps understand order fulfillment distribution
order_by_status_count = customers_orders_df.groupBy('status').count().orderBy(col('count').desc())
order_by_status_count.show(10)

# Note: The line below appears incomplete (top_products analysis)
top_products = customers_orders_df.count

In [0]:
# Analyze order volume by month to identify seasonal trends
# Results are ordered chronologically (month 1-12)
order_by_month = customers_orders_df.groupBy('order_month').count().orderBy(col('order_month').asc())
order_by_month.show(10)

In [0]:
# Rank customers by total spend using dense_rank
# Dense rank assigns consecutive ranks without gaps (useful for top-N analysis)
window_spec = Window.orderBy(col('total_spend').desc())
dense_ranked_customers = customer_total_spend.withColumn('dense_rank', dense_rank().over(window_spec))
dense_ranked_customers.show(5)

In [0]:
# Identify customers with high order frequency but low total spending
# These are customers who order often but spend less per order (potential upsell targets)
customer_spend_vs_order = customers_orders_count.join(
    customer_total_spend, 'customer_id', 'inner'
).orderBy(col('count').desc(), col('total_spend'))
customer_spend_vs_order.show(5)